# Notebook 2: Data Preprocessing & Augmentation
This notebook details our preprocessing and data augmentation pipeline.
We demonstrate how to split raw data and augment scans using `ImageDataGenerator`.


In [ ]:
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# Import our preprocessing split function
import sys
sys.path.append('..')
from ml.preprocessing import split_dataset

BASE_DIR = Path('../dataset')
PROCESSED_DIR = BASE_DIR / 'processed'
print('Processed folder:', PROCESSED_DIR.resolve())


### Step 1: Split and Reorganize Dataset
If you haven't run the split script, let's run it here. It combines validation and test sets and performs a stratified 70/15/15 split.


In [ ]:
if not PROCESSED_DIR.exists():
    split_dataset(BASE_DIR, PROCESSED_DIR)
else:
    print('Processed dataset already split and ready.')


### Step 2: Configure Image Data Generator (Data Augmentation)
Augmentations help prevent overfitting on small datasets.


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

print('Training generator configured with Rescaling, Rotations, Shifts, Zoom, and Flips.')


### Step 3: Visualize Augmented Images for a Single Scan


In [ ]:
# Pick a random image
sample_path = list((PROCESSED_DIR / 'train' / 'glioma').glob('*'))[0]
img = load_img(sample_path, target_size=(224, 224))
x = img_to_array(img) / 255.0
x = np.expand_dims(x, axis=0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img)
axes[0].set_title('Original Image', fontweight='bold')
axes[0].axis('off')

# Generate 3 random augmented versions
i = 1
for batch in train_datagen.flow(x, batch_size=1):
    axes[i].imshow(batch[0])
    axes[i].set_title(f'Augmented {i}', fontweight='bold')
    axes[i].axis('off')
    i += 1
    if i > 3:
        break
plt.tight_layout()
plt.show()
